# Portion 2 · TabBERT encoder: pretraining and handoffs

Pretrain the two-level TabBERT encoder with masked field modelling on train-period windows, then export embeddings, fine-tuning scores and reconstruction scores for Portions 3 and 4. Design: [`PORTION2_BUILD_GUIDE.md`](PORTION2_BUILD_GUIDE.md). Step-by-step plan: [`PORTION2_IMPLEMENTATION_PLAN.md`](PORTION2_IMPLEMENTATION_PLAN.md).

Run everything on a Colab GPU runtime (L4 or better). Checkpoints and outputs go to Google Drive, not to Git. Every step skips work already saved on Drive, so the notebook can be rerun from the top after a disconnect.

## 1. Get the code and the Portion 1 data

The first code cell clones the repository and pulls only the Portion 1 files this portion reads through Git LFS (about 190 MB instead of the full 530 MB), which keeps LFS bandwidth low. When the notebook runs from a local checkout, it uses that checkout instead.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/ted-cc6/TabBERT_AML"
BRANCH = "main"
LFS_FILES = [
    "Portion1/processed/event_tokens.npz",
    "Portion1/processed/window_index.csv.gz",
    "Portion1/processed/events_meta.csv.gz",
    "Portion1/processed/label_subsets.csv.gz",
    "Portion1/processed/label_subset_100pct.csv.gz",
]

current = Path.cwd()
PORTION2_DIR = next((p for p in (current, current / "Portion2") if (p / "portion2_encoder.py").is_file()), None)

if PORTION2_DIR is None:
    clone_root = Path("/content/aml_project_repo")
    if shutil.which("git-lfs") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=True)
    subprocess.run(["git", "lfs", "install", "--skip-smudge"], check=True)
    if not clone_root.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(clone_root)], check=True)
    else:
        subprocess.run(["git", "-C", str(clone_root), "pull", "--ff-only"], check=True)
    subprocess.run(["git", "-C", str(clone_root), "lfs", "pull", "--include", ",".join(LFS_FILES)], check=True)
    PORTION2_DIR = clone_root / "Portion2"

PROCESSED_DIR = PORTION2_DIR.parent / "Portion1" / "processed"
sys.path.insert(0, str(PORTION2_DIR))
for name in LFS_FILES:
    size = (PORTION2_DIR.parent / name).stat().st_size
    assert size > 1_000_000, f"{name} is still an LFS pointer ({size} bytes)"
print(PORTION2_DIR)

## 2. Check the GPU and mount Drive

`OUT_DIR` is the shared Drive folder for checkpoints, metrics, embeddings and score files. Each run writes to `OUT_DIR/<run_name>/`.

In [ ]:
import importlib

import torch

import portion2_data
import portion2_encoder as enc

# Reload so code pulled by cell 1 takes effect even when this Python session already imported an older version.
importlib.reload(portion2_data)
importlib.reload(enc)

DEVICE = enc.pick_device()
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
if DEVICE.type != "cuda":
    print("No CUDA GPU: switch the runtime to a GPU before the full run.")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/TabBERT_AML/portion2_runs")
except ImportError:
    OUT_DIR = PORTION2_DIR / "runs"  # local runs; ignored by Git
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs:", OUT_DIR)

## 2b. Run the tests

Runs the Portion 2 test suite on this runtime: CPU tests on the real Portion 1 data, plus GPU smoke tests that run export, reconstruction and fine-tuning on CUDA at real batch sizes. Takes about 5 minutes. If anything fails, the cell stops; paste the output and do not continue.

In [ ]:
import importlib.util

if importlib.util.find_spec("pytest") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"],
                        cwd=PORTION2_DIR, capture_output=True, text=True)
print(result.stdout[-4000:])
print(result.stderr[-2000:])
assert result.returncode == 0, "Tests failed: paste the output above"

## 3. Main pretraining, seed 42 (Task 4)

The adopted model (4 sequence layers, d_model 256, 4.29M parameters; see section 5), field-level masking with Δt, 1M train windows per epoch, 3 epochs, batch 512, AdamW 1e-3 with warmup and cosine decay. About 7 minutes on an L4. Each run writes `checkpoint.pt`, `metrics.csv` and `amount_embedding_cosine.png` to `OUT_DIR/<run_name>/`.

`run_or_load` skips training when the run's checkpoint is already on Drive, and stops with an error if the saved run used different settings. Set `QUICK_CHECK = True` to test a new runtime on 50K windows (about a minute).

In [ ]:
import dataclasses
import json
import time

QUICK_CHECK = False


def run_or_load(config_name, **overrides):
    cfg = enc.load_run_config(PORTION2_DIR / "configs" / f"{config_name}.json")
    if overrides:
        cfg = dataclasses.replace(cfg, **overrides)
    ckpt = OUT_DIR / cfg.run_name / "checkpoint.pt"
    if ckpt.exists():
        saved = json.loads((ckpt.parent / "run_config.json").read_text())
        if saved != dataclasses.asdict(cfg):
            raise RuntimeError(f"{ckpt.parent} holds a run with different settings. Rename that Drive folder "
                               f"(for example to {cfg.run_name}_old), then rerun.")
        print("Already trained:", ckpt)
        return ckpt
    start = time.time()
    ckpt = enc.pretrain(cfg, PROCESSED_DIR, OUT_DIR, device=DEVICE, log_every=200)
    print(f"Done in {(time.time() - start) / 60:.1f} min:", ckpt)
    return ckpt


if QUICK_CHECK:
    main_ckpt = run_or_load("main_seed42", run_name="quick_check", windows_per_epoch=50_000, epochs=1,
                            valid_windows=5_000, warmup_steps=20)
else:
    main_ckpt = run_or_load("main_seed42")

## 4. Check the result

Done when each field's masked accuracy (`acc_`) is clearly above its most-frequent-token baseline (`baseline_`), especially for currency, counterparty bank, amount and time gap. Direction at the last position is excluded, because the target event is always "sent". The heatmap should show neighbouring amount buckets as similar (a bright band along the diagonal).

In [ ]:
import pandas as pd
from IPython.display import Image, display


def summarize(ckpt_path):
    metrics = pd.read_csv(ckpt_path.parent / "metrics.csv")
    _, specs, _ = enc.load_encoder(ckpt_path, torch.device("cpu"))
    fields = [s.name for s in specs]
    summary = pd.DataFrame({
        "accuracy": [metrics[f"acc_{f}"].iloc[-1] for f in fields],
        "baseline": [metrics[f"baseline_{f}"].iloc[-1] for f in fields],
    }, index=fields)
    summary["lift"] = summary.accuracy - summary.baseline
    extra = [c for c in metrics.columns if c.startswith("acc1_") or c.endswith("_nonzero")]
    print(metrics[["epoch", "train_loss", "valid_loss", "seconds"]].to_string(index=False), "\n")
    print(summary.round(3).to_string(), "\n")
    print(metrics[["epoch"] + extra].round(3).to_string(index=False))
    display(Image(filename=str(ckpt_path.parent / "amount_embedding_cosine.png")))


summarize(main_ckpt)

## 5. Model size decision (record)

On 2026-11-20 the original 1.5M model (`small_seed42`: 2 layers, d_model 128) was compared with a 4.29M model (`capacity_check_seed42`: 4 layers, d_model 256), same data, seed and schedule. Rule fixed before the run: keep the larger model only if its validation masked loss is at least 5% lower. It was 17.4% lower (0.669 against 0.810), so all main and ablation runs use the larger model. This cell only reloads both saved runs and prints the comparison; it trains nothing.

In [ ]:
def compare(ckpts):
    table = {}
    for ckpt in ckpts:
        model, specs, cfg = enc.load_encoder(ckpt, torch.device("cpu"))
        last = pd.read_csv(ckpt.parent / "metrics.csv").iloc[-1]
        table[ckpt.parent.name] = {
            "parameters": sum(p.numel() for p in model.parameters()),
            "valid_loss": last.valid_loss,
            "train_loss": last.train_loss,
            "minutes": last.seconds / 60,
            **{f"acc_{s.name}": last[f"acc_{s.name}"] for s in specs},
        }
    return pd.DataFrame(table)


size_runs = [OUT_DIR / name / "checkpoint.pt" for name in ("small_seed42", "capacity_check_seed42")]
if all(p.exists() for p in size_runs):
    comparison = compare(size_runs)
    print(comparison.round(4).to_string())
    gain = 1 - comparison.loc["valid_loss"].iloc[1] / comparison.loc["valid_loss"].iloc[0]
    print(f"\nValidation loss reduction from the larger model: {gain:.1%} (rule: keep it only if >= 5%)")
else:
    print("Size-check runs not found on Drive:", [str(p) for p in size_runs if not p.exists()])

## 6. Export frozen embeddings (Task 5)

One embedding per transaction for Portion 3: the target-row vector (first 256 columns), then the mean over the window's real rows (last 256), so 512 float16 values. If Portion 3 wants fewer, the first 256 columns are the target-only version. Rows follow `window_index.csv.gz` order and cover all splits including tail; the `_ids.npy` file holds the `transaction_id` of each row, so join on that.

The file is about 5.2 GB. It is written to the runtime's local disk first, then copied to Drive (writing a memory-mapped file straight to Drive is slow). Skipped if the files are already on Drive.

In [ ]:
import numpy as np

EXPORT_RUN = "main_seed42"
target_dir = OUT_DIR / EXPORT_RUN
emb_name, ids_name = f"embeddings_{EXPORT_RUN}.npy", f"embeddings_{EXPORT_RUN}_ids.npy"

if (target_dir / emb_name).exists() and (target_dir / ids_name).exists():
    print("Already exported:", target_dir / emb_name)
else:
    local_dir = Path("/content/exports") if Path("/content").exists() else OUT_DIR / "exports"
    emb_path, ids_path = enc.export_embeddings(target_dir / "checkpoint.pt", PROCESSED_DIR, local_dir, device=DEVICE)
    if local_dir != target_dir:
        for path in (emb_path, ids_path):
            shutil.copyfile(path, target_dir / path.name)  # copy2 also copies timestamps, which some mounts refuse
            print("Copied to", target_dir / path.name)

embeddings = np.load(target_dir / emb_name, mmap_mode="r")
ids = np.load(target_dir / ids_name)
print(embeddings.shape, embeddings.dtype, f"{embeddings.nbytes / 1e9:.2f} GB")
print("ids:", len(ids), "unique:", len(np.unique(ids)))
print("finite in first 100K rows:", bool(np.isfinite(embeddings[:100_000]).all()))

## 7. Fine-tuning (Task 6)

Adds a small MLP head on the target-row vector and trains encoder and head end to end on one labelled train subset only (unlabelled train transactions are not negatives). Class-weighted loss; each epoch uses every positive plus at most 200,000 negatives; at most 10 epochs, early stopping after 3 epochs without a gain in validation PR AUC. Writes `scores.csv` (`transaction_id, split, score` for valid and test, from the best epoch), `finetune_log.csv` and `summary.json` to `OUT_DIR/finetune/ft_<run>_<pct>pct_seed<seed>/`. Test is scored but never used for any choice.

`FT_RUNS` lists (pretraining run, label fraction, downstream seed). Downstream seed s uses the main checkpoint pretrained with seed s; ablations use their seed-42 checkpoint with seeds 42, 43 and 44. Finished runs are skipped.

In [ ]:
FT_RUNS = [
    ("main_seed42", 0.05, 42),
]

FT_DIR = OUT_DIR / "finetune"
for run, fraction, seed in FT_RUNS:
    done = FT_DIR / f"ft_{run}_{fraction * 100:g}pct_seed{seed}" / "scores.csv"
    if done.exists():
        print("Already done:", done)
        continue
    enc.finetune(OUT_DIR / run / "checkpoint.pt", PROCESSED_DIR, FT_DIR, fraction=fraction, seed=seed, device=DEVICE)

summaries = [json.loads(p.read_text()) for p in sorted(FT_DIR.glob("ft_*/summary.json"))]
print(pd.DataFrame(summaries).to_string(index=False))

## 8. Reconstruction score (Task 7)

Zero-shot anomaly score for Portion 4. For each valid, test and tail transaction, the encoder sees the window five times, each time with one field of the target transaction masked (payment format, currency, counterparty bank, amount, time gap; direction is skipped), and records how surprised it is by the true value (negative log likelihood). Output `recon_<run>.csv`: `transaction_id, split`, one `nll_<field>` column per field and `nll_sum`. Portion 4 normalises per field. About 1.83M transactions; written locally, then copied to Drive.

The check at the end compares laundering and normal transactions on validation only: a higher mean score and a PR AUC above the positive rate mean the score carries some signal before any labels are used.

In [ ]:
from sklearn.metrics import average_precision_score

RECON_RUNS = ["main_seed42"]

for run in RECON_RUNS:
    target = OUT_DIR / run / f"recon_{run}.csv"
    if target.exists():
        print("Already done:", target)
    else:
        local_dir = Path("/content/recon") if Path("/content").exists() else OUT_DIR / "recon"
        path = enc.reconstruction_scores(OUT_DIR / run / "checkpoint.pt", PROCESSED_DIR, local_dir, device=DEVICE)
        if path != target:
            shutil.copyfile(path, target)
            print("Copied to", target)

    recon = pd.read_csv(target)
    labels = pd.read_csv(PROCESSED_DIR / "window_index.csv.gz", usecols=["transaction_id", "is_laundering"])
    valid = recon[recon.split == "valid"].merge(labels, on="transaction_id")
    nll_cols = [c for c in recon.columns if c.startswith("nll_")]
    print(recon.groupby("split").size().to_string(), "\n")
    print("Mean score on valid, by label:")
    print(valid.groupby("is_laundering")[nll_cols].mean().round(3).to_string(), "\n")
    print(f"Valid PR AUC of nll_sum: {average_precision_score(valid.is_laundering, valid.nll_sum):.4f} "
          f"(positive rate {valid.is_laundering.mean():.4f})")